# 0. Import Library

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold

# 1. Load Dataset & Split Dataset

## 1.1 Load Dataset

In [ ]:
# Load data
df = pd.read_csv("../data/stock_prices.csv")

# Kolom yang akan dipertahankan
keep_cols = ["RowId", "Date", "SecuritiesCode", "Open", "High", "Low", "Close", "Volume", "Target"]
df = df[keep_cols].copy()

# Format kolom "Date" menjadi tipe datetime
df["Date"] = pd.to_datetime(df["Date"])

# Memastikan urutan data berdasarkan tanggal
df = df.sort_values("Date").reset_index(drop=True)

print(f"Total rows: {len(df)}")
print(f"Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")

Total rows: 4717106
Date range: 2017-01-04 → 2021-12-03


: 

## 1.2 Timeseries Split

In [ ]:
# Split berdasarkan tanggal, 80% untuk train dan 20% untuk test
split_date = df["Date"].quantile(0.8)  

train = df[df["Date"] <= split_date].copy()
test  = df[df["Date"] >  split_date].copy()

print(f"\nTrain: {train['Date'].min().date()} → {train['Date'].max().date()}  ({len(train)} rows)")
print(f"Test : {test['Date'].min().date()} → {test['Date'].max().date()}  ({len(test)} rows)")

train.to_csv("train.csv", index=True)
test.to_csv("test.csv", index=True)

# 2. Missing Values

## 2.1 Load Dataset After Split

In [ ]:
# Load train and test data 
BASE_DIR = Path.cwd()
train = pd.read_csv(BASE_DIR / "train.csv")
test  = pd.read_csv(BASE_DIR / "test.csv")

# Re-parse kolom "Date"
train["Date"] = pd.to_datetime(train["Date"])
test["Date"]  = pd.to_datetime(test["Date"])

print(f"\ntrain: {train.shape}")
print(f"test : {test.shape}")
train.head()

## 2.2 Dataset Shape & Dtypes

In [ ]:
print(f"Shape: {train.shape}")
print("\nDtypes:")
print(train.dtypes)

## 2.3 Missing Values in Train & Test

In [ ]:
missing = pd.DataFrame({
    "missing_count": train.isna().sum(),
    "missing_pct":   (train.isna().mean() * 100).round(4),
    "dtype":         train.dtypes.astype(str),
}).sort_values("missing_count", ascending=False)
print(missing)

In [ ]:
missing_test = pd.DataFrame({
    "missing_count": test.isna().sum(),
    "missing_pct":   (test.isna().mean() * 100).round(4),
}).sort_values("missing_count", ascending=False)
print(missing_test)

## 2.4 Missing Values in Train

In [ ]:
# ============================================================
# 2.4 Missing Values in Train
# ============================================================

total_cells           = train.shape[0] * train.shape[1]
total_missing         = train.isna().sum().sum()
rows_with_any_missing = train.isna().any(axis=1).sum()
rows_with_all_missing = train.isna().all(axis=1).sum()

print(f"Total cells           : {total_cells}")
print(f"Total missing cells   : {total_missing}  ({total_missing / total_cells * 100:.4f}%)")
print(f"Rows w/ >=1 missing   : {rows_with_any_missing}  ({rows_with_any_missing / len(train) * 100:.4f}%)")
print(f"Rows w/ ALL missing   : {rows_with_all_missing}")

for col in train.columns:
    if train[col].isna().any():
        idx = train.index[train[col].isna()]
        print(f"{col:16s} | first @row {idx.min():>7} | last @row {idx.max():>7} | n_missing={len(idx)}")
    else:
        print(f"{col:16s} | no missing values")


## 2.5 Missing Values by Year

In [ ]:
train["_year"] = train["Date"].dt.year
print(train.groupby("_year").apply(lambda g: g.isna().sum()))
train.drop(columns="_year", inplace=True)

## 2.6 Missng Values by Security

In [ ]:
per_sec = train.groupby("SecuritiesCode").apply(lambda g: g.isna().sum())
per_sec = per_sec[per_sec.sum(axis=1) > 0]
if per_sec.empty:
    print("No security has any missing values.")
else:
    print(f"{per_sec.shape[0]} securities have missing values. Top 10:")
    print(per_sec.assign(total=per_sec.sum(axis=1))
                  .sort_values("total", ascending=False).head(10))

## 2.7 Numeric Distribution

In [ ]:
num_cols = ["Open", "High", "Low", "Close", "Volume", "Target"]
print(train[num_cols].describe().T)


## 2.8 Skewness Data

In [ ]:
print(train[num_cols].skew().round(4))

# 3. Implementation — Fit on Train, Apply to Test

## 3.1 Confirm Missing OHLC Pattern

In [ ]:
ohlc_cols = ["Open", "High", "Low", "Close"]

for name, df_ in [("train", train), ("test", test)]:
    mask = df_[ohlc_cols].isna().any(axis=1)
    n = mask.sum()
    n_zero_vol = (df_.loc[mask, "Volume"] == 0).sum()
    print(f"{name}: OHLC-missing rows = {n}, of which Volume==0 = {n_zero_vol} "
          f"({n_zero_vol/n*100:.2f}%)" if n else f"{name}: no OHLC-missing rows")


    drop_subset = ["Open", "High", "Low", "Close", "Target"]

train_clean = train.dropna(subset=drop_subset).reset_index(drop=True)
test_clean  = test.dropna(subset=drop_subset).reset_index(drop=True)

print(f"\nBefore: train={train.shape}  test={test.shape}")
print(f"After : train={train_clean.shape}  test={test_clean.shape}")
print(f"Dropped train rows: {len(train) - len(train_clean)} "
      f"({(len(train)-len(train_clean))/len(train)*100:.4f}%)")
print(f"Dropped test  rows: {len(test)  - len(test_clean)} "
      f"({(len(test)-len(test_clean))/len(test)*100:.4f}%)")



## 3.2 Post-cleanup Verification

In [ ]:
print("\nRemaining missing values — train:", int(train_clean.isna().sum().sum()))
print("Remaining missing values — test :", int(test_clean.isna().sum().sum()))

# Any security fully wiped out?
before_secs = set(train["SecuritiesCode"].unique())
after_secs  = set(train_clean["SecuritiesCode"].unique())
lost = before_secs - after_secs
print(f"Securities present in train before: {len(before_secs)}  |  after: {len(after_secs)}")
if lost:
    print(f"  ⚠ {len(lost)} securities fully removed: {sorted(lost)[:20]}{'...' if len(lost) > 20 else ''}")
else:
    print("  ✓ No security was completely removed.")

# Sanity: Volume should now be > 0 wherever OHLC exists (mostly)
print("\nVolume==0 rows remaining in train_clean:", int((train_clean["Volume"] == 0).sum()))
print("Volume==0 rows remaining in test_clean :", int((test_clean["Volume"]  == 0).sum()))


# 4. Encoding Diagnostic

## 4.1 Confirm Clean Data and Column Types

### Ringkasan Bentuk & Missing Value

In [ ]:
train = train_clean
test  = test_clean

print(f"train shape : {train.shape}")
print(f"test shape  : {test.shape}")
print(f"train NaN   : {int(train.isna().sum().sum())}")
print(f"test  NaN   : {int(test.isna().sum().sum())}")


### Tipe Data, Kardinalitas, dan Kolom Kategorikal

In [ ]:
info = pd.DataFrame({
    "dtype":        train.dtypes.astype(str),
    "n_unique":     train.nunique(dropna=False),
    "n_rows":       len(train),
    "unique_ratio": (train.nunique(dropna=False) / len(train)).round(6),
})
print(info)

for col in train.columns:
    dt    = train[col].dtype
    n     = train[col].nunique(dropna=False)
    ratio = n / len(train)
    is_cat_dtype = str(dt) in ("object", "string", "str", "category")
    tag = []
    if is_cat_dtype:
        tag.append("dtype=categorical")
    if ratio > 0.95 and not str(dt).startswith("datetime"):
        tag.append("IDENTIFIER (nearly unique)")
    if 1 < n <= 50:
        tag.append("low-cardinality")
    print(f"{col:16s} | dtype={str(dt):15s} | nunique={n:>8} | ratio={ratio:.4f} | "
          f"{' , '.join(tag) if tag else '-'}")

obj_cols = train.select_dtypes(include=["object", "string", "str", "category"]).columns.tolist()
if not obj_cols:
    print("No object/string/category columns found.")
else:
    for col in obj_cols:
        print(f"\n--- {col} ---")
        print(f"dtype        : {train[col].dtype}")
        print(f"n_unique     : {train[col].nunique()}")
        print(f"top 5 values : {train[col].value_counts().head(5).to_dict()}")
        print(f"most common  : {train[col].mode().iloc[0] if not train[col].mode().empty else 'N/A'}")
        print(f"sample       : {train[col].dropna().unique()[:5]}")


### Validasi RowId & SecuritiesCode

In [ ]:
print(f"n_unique RowId       : {train['RowId'].nunique()}")
print(f"n_rows               : {len(train)}")
print(f"RowId fully unique?  : {train['RowId'].is_unique}")
print(f"sample RowIds        : {train['RowId'].head(3).tolist()}")
reconstructed = train["Date"].dt.strftime("%Y%m%d") + "_" + train["SecuritiesCode"].astype(str)
print(f"RowId == 'YYYYMMDD_Code'? : {(reconstructed == train['RowId']).mean():.4%} match")

all_codes = np.sort(train["SecuritiesCode"].unique())
gaps = np.diff(all_codes)
print(f"codes contiguous?            : {(gaps == 1).all()}")
print(f"largest gap between codes    : {int(gaps.max()) if len(gaps) else 0}")
rows_per_sec = train["SecuritiesCode"].value_counts()
print(f"rows per security: min={rows_per_sec.min()}, median={int(rows_per_sec.median())}, max={rows_per_sec.max()}")
print(f"securities with < 30 rows    : {int((rows_per_sec < 30).sum())}")

train_secs = set(train["SecuritiesCode"].unique())
test_secs  = set(test["SecuritiesCode"].unique())
print(f"Securities in train           : {len(train_secs)}")
print(f"Securities in test            : {len(test_secs)}")
print(f"In test but NOT in train      : {len(test_secs - train_secs)}")
if test_secs - train_secs:
    print(f"  examples: {sorted(test_secs - train_secs)[:10]}")
print(f"In train but NOT in test      : {len(train_secs - test_secs)}")


### Rentang Tanggal, Distribusi Target, & Duplikasi

In [ ]:


print(f"train Date range : {train['Date'].min().date()} -> {train['Date'].max().date()}")
print(f"test  Date range : {test['Date'].min().date()}  -> {test['Date'].max().date()}")
print(f"n unique dates   : train={train['Date'].nunique()}, test={test['Date'].nunique()}")
tmp = train.assign(_year=train["Date"].dt.year)
print("trading days per year:")
print(tmp.groupby("_year")["Date"].nunique())
del tmp


grp = train.groupby("SecuritiesCode")["Target"].agg(["mean", "std", "count"])
print(f"Overall Target mean/std          : {train['Target'].mean():.6f} / {train['Target'].std():.6f}")
print(f"Per-security Target mean std     : {grp['mean'].std():.6f}")
print(f"Per-security Target mean min/max : {grp['mean'].min():.6f} / {grp['mean'].max():.6f}")

dup = train.duplicated(subset=["Date", "SecuritiesCode"]).sum()
print(f"Duplicate (Date, SecuritiesCode) rows in train: {int(dup)}")

print("\n" + "=" * 70)
print("10. NUMERIC COLUMNS — confirm nothing categorical hides here")
print("=" * 70)
num_cols = ["Open", "High", "Low", "Close", "Volume", "Target"]
print(f"All numeric? {all(pd.api.types.is_numeric_dtype(train[c]) for c in num_cols)}")
print(train[num_cols].dtypes)


## 4.2 Encoding for Securities Code

### Tahap 0 — Restart dari Cleaned Frames

In [ ]:
from sklearn.model_selection import KFold

train = train_clean.copy()
test  = test_clean.copy()
print(f"Before encoding: train={train.shape}  test={test.shape}")


### Tahap 1 — Drop RowId

In [ ]:
train = train.drop(columns=["RowId"])
test  = test.drop(columns=["RowId"])

### Tahap 2 — Date → Fitur Temporal

In [ ]:
for df in (train, test):
    df["year"]       = df["Date"].dt.year
    df["month"]      = df["Date"].dt.month
    df["day"]        = df["Date"].dt.day
    df["dayofweek"]  = df["Date"].dt.dayofweek
    df["quarter"]    = df["Date"].dt.quarter
    df["dayofyear"]  = df["Date"].dt.dayofyear
    df["weekofyear"] = df["Date"].dt.isocalendar().week.astype(int)



### Tahap 3 — SecuritiesCode → Smoothed Target Encoding (leak-free)

In [ ]:
K = 50                                
global_mean = train["Target"].mean()  

train = train.sort_values("Date").reset_index(drop=True)
train["SecuritiesCode_te"] = np.nan

# 3a) Out-of-fold encoding for train (each row encoded without its own target)
kf = KFold(n_splits=5, shuffle=False)     
for tr_idx, va_idx in kf.split(train):
    sub = train.iloc[tr_idx]
    agg = sub.groupby("SecuritiesCode")["Target"].agg(["mean", "count"])
    smoothed = (agg["count"] * agg["mean"] + K * global_mean) / (agg["count"] + K)
    train.loc[va_idx, "SecuritiesCode_te"] = (
        train.iloc[va_idx]["SecuritiesCode"].map(smoothed).values
    )

train["SecuritiesCode_te"] = train["SecuritiesCode_te"].fillna(global_mean).astype(np.float32)

# 3b) Full-train mapping for test (unseen → global mean)
agg_full = train.groupby("SecuritiesCode")["Target"].agg(["mean", "count"])
smoothed_full = (agg_full["count"] * agg_full["mean"] + K * global_mean) / (agg_full["count"] + K)
test["SecuritiesCode_te"] = (
    test["SecuritiesCode"].map(smoothed_full).fillna(global_mean).astype(np.float32)
)



### Tahap 4 — Drop Raw Identifiers

In [ ]:
train = train.drop(columns=["SecuritiesCode", "Date"])
test  = test.drop(columns=["SecuritiesCode", "Date"])

### Tahap 5 — Verifikasi Hasil Akhir

In [ ]:

print("\n" + "=" * 60)
print("FINAL SHAPES")
print("=" * 60)
print(f"train: {train.shape}")
print(f"test : {test.shape}")
print(f"train NaN: {int(train.isna().sum().sum())}")
print(f"test  NaN: {int(test.isna().sum().sum())}")
print(f"All numeric? {all(pd.api.types.is_numeric_dtype(train[c]) for c in train.columns)}")

print("\nFINAL COLUMNS")
print(list(train.columns))

print("\nSecuritiesCode_te spread (train):")
print(train["SecuritiesCode_te"].describe())
print(f"unique encoded values: {train['SecuritiesCode_te'].nunique()}")

print("\nHEAD")
print(train.head())


# 5. Feature Engineering Diagnostic

## 5.1 Rebuild from Cleaned Frames

In [ ]:
train = train_clean.copy()
test  = test_clean.copy()

K = 50
global_mean = train["Target"].mean()
print(f"Global mean of Target: {global_mean:.6f}")

train = train.sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)
test  = test.sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)

for df in (train, test):
    df["year"]       = df["Date"].dt.year
    df["month"]      = df["Date"].dt.month
    df["day"]        = df["Date"].dt.day
    df["dayofweek"]  = df["Date"].dt.dayofweek
    df["quarter"]    = df["Date"].dt.quarter
    df["dayofyear"]  = df["Date"].dt.dayofyear
    df["weekofyear"] = df["Date"].dt.isocalendar().week.astype(int)

# ---- Leak-free out-of-fold target encoding for train ----
train["SecuritiesCode_te"] = np.nan
kf = KFold(n_splits=5, shuffle=False)
for tr_idx, va_idx in kf.split(train):
    sub = train.iloc[tr_idx]
    agg = sub.groupby("SecuritiesCode")["Target"].agg(["mean", "count"])
    sm  = (agg["count"] * agg["mean"] + K * global_mean) / (agg["count"] + K)
    train.loc[va_idx, "SecuritiesCode_te"] = (
        train.iloc[va_idx]["SecuritiesCode"].map(sm).values
    )
train["SecuritiesCode_te"] = train["SecuritiesCode_te"].fillna(global_mean).astype(np.float32)

# ---- Full-train mapping for test (unseen → global mean) ----
agg_full = train.groupby("SecuritiesCode")["Target"].agg(["mean", "count"])
sm_full  = (agg_full["count"] * agg_full["mean"] + K * global_mean) / (agg_full["count"] + K)
test["SecuritiesCode_te"] = (
    test["SecuritiesCode"].map(sm_full).fillna(global_mean).astype(np.float32)
)

print(f"Rebuilt with keys kept: train={train.shape}  test={test.shape}")
print(f"Columns: {list(train.columns)}")


### Fungsi Bantu — Pengecekan Urutan (is_sorted_by)

In [ ]:
def is_sorted_by(df, keys):
    if len(df) < 2:
        return True
    cols = []
    for k in keys:
        s = df[k]
        if pd.api.types.is_datetime64_any_dtype(s):
            cols.append(s.astype("int64").to_numpy())
        else:
            cols.append(s.to_numpy())
    arr = np.column_stack(cols).astype("int64")
    return bool((np.diff(arr, axis=0) >= 0).all())

### Diagnostic 1 — Urutan Waktu & Pengelompokan

In [ ]:
print(f"train sorted by (Date, Code)? {is_sorted_by(train, ['Date','SecuritiesCode'])}")
print(f"test  sorted by (Date, Code)? {is_sorted_by(test,  ['Date','SecuritiesCode'])}")
print(f"Duplicate (Date, Code) rows in train: "
      f"{int(train.duplicated(['Date','SecuritiesCode']).sum())}")
per_sec_days = train.groupby("SecuritiesCode")["Date"].nunique()
print(f"Days per security: min={per_sec_days.min()}, "
      f"median={int(per_sec_days.median())}, max={per_sec_days.max()}")



### Diagnostic 2 — Distribusi Return

In [ ]:
train["_ret"] = train.groupby("SecuritiesCode")["Close"].pct_change()
print(train["_ret"].describe(percentiles=[0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999]))
print(f"Extreme returns |r|>0.2 : {int((train['_ret'].abs() > 0.2).sum())}")
print(f"Extreme returns |r|>0.5 : {int((train['_ret'].abs() > 0.5).sum())}")



### Diagnostic 3 — Autokorelasi Return

In [ ]:
def lag1_autocorr(s):
    s = s.dropna()
    return s.autocorr(lag=1) if len(s) > 3 else np.nan

autocorrs = train.groupby("SecuritiesCode")["_ret"].apply(lag1_autocorr).dropna()
print(f"Mean lag-1 autocorr : {autocorrs.mean():.5f}")
print(f"Std  lag-1 autocorr : {autocorrs.std():.5f}")
print(f"% securities with |autocorr| > 0.05 : "
      f"{(autocorrs.abs() > 0.05).mean()*100:.2f}%")



### Diagnostic 4 — Distribusi Volume

In [ ]:
print(f"Volume skew (raw)   : {train['Volume'].skew():.3f}")
print(f"Volume skew (log1p) : {np.log1p(train['Volume']).skew():.3f}")
print(f"Volume == 0 count   : {int((train['Volume'] == 0).sum())}")

### Diagnostic 5 — Sebaran Cross-sectional per Hari

In [ ]:
daily_std = train.groupby("Date")["_ret"].std()
print(f"Daily cross-sectional return std: mean={daily_std.mean():.5f}, "
      f"median={daily_std.median():.5f}, max={daily_std.max():.5f}")

### Diagnostic 6 — Korelasi Pearson dengan Target & Cleanup

In [ ]:
train["_hl"]       = (train["High"] - train["Low"]) / train["Close"]
train["_co"]       = (train["Close"] - train["Open"]) / train["Open"]
train["_logvol"]   = np.log1p(train["Volume"])
train["_ret_lag1"] = train.groupby("SecuritiesCode")["_ret"].shift(1)

candidates = ["_hl", "_co", "_logvol", "_ret", "_ret_lag1", "SecuritiesCode_te",
              "year", "month", "dayofweek", "quarter"]
corr = train[candidates + ["Target"]].corr()["Target"].drop("Target")
print(corr.sort_values(ascending=False).round(5))

# Cleanup scratch cols
train = train.drop(columns=["_ret", "_hl", "_co", "_logvol", "_ret_lag1"])
print("\nScratch columns removed. Final train shape:", train.shape)


## 5.2 Build FE on Train + Test Concatenated

In [ ]:
train = train_clean.copy()
test  = test_clean.copy()

train["_split"] = "train"
test["_split"]  = "test"
full = pd.concat([train, test], axis=0, ignore_index=True)
full = full.sort_values(["SecuritiesCode", "Date"]).reset_index(drop=True)
print(f"Concat: {full.shape}")

g = full.groupby("SecuritiesCode", sort=False)

full["ret_1d"]  = g["Close"].pct_change()
full["ret_5d"]  = g["Close"].pct_change(5)
full["ret_20d"] = g["Close"].pct_change(20)

full["intraday_range"] = (full["High"] - full["Low"]) / full["Close"]
full["close_open"]     = (full["Close"] - full["Open"]) / full["Open"]
full["high_close"]     = (full["High"] - full["Close"]) / full["Close"]
full["low_close"]      = (full["Close"] - full["Low"]) / full["Close"]

full["log_volume"]   = np.log1p(full["Volume"])
vol_ma20             = g["Volume"].transform(lambda s: s.rolling(20, min_periods=1).mean())
full["volume_ratio"] = full["Volume"] / vol_ma20
del vol_ma20

full["vol_5d"]  = g["ret_1d"].transform(lambda s: s.rolling(5,  min_periods=2).std())
full["vol_20d"] = g["ret_1d"].transform(lambda s: s.rolling(20, min_periods=5).std())

full["ret_rank"]    = full.groupby("Date")["ret_1d"].rank(pct=True)
full["volume_rank"] = full.groupby("Date")["Volume"].rank(pct=True)

full = full.drop(columns=["Open", "High", "Low", "Close", "Volume"])

train_fe = full[full["_split"] == "train"].drop(columns="_split").reset_index(drop=True)
test_fe  = full[full["_split"] == "test"].drop(columns="_split").reset_index(drop=True)

print(f"Before NaN drop: train={train_fe.shape}  test={test_fe.shape}")
train_fe = train_fe.dropna().reset_index(drop=True)
test_fe  = test_fe.dropna().reset_index(drop=True)
print(f"After  NaN drop: train={train_fe.shape}  test={test_fe.shape}")

train_fe = train_fe.drop(columns=["RowId"])
test_fe  = test_fe.drop(columns=["RowId"])

### Ringkasan Hasil Feature Engineering

In [ ]:
print(f"\nFinal train_fe columns ({len(train_fe.columns)}):")
print(list(train_fe.columns))
print(f"\nNaN in train_fe: {int(train_fe.isna().sum().sum())}")
print(f"NaN in test_fe : {int(test_fe.isna().sum().sum())}")
print(f"\ntrain_fe shape: {train_fe.shape}")
print(f"test_fe  shape: {test_fe.shape}")
print(f"All numeric?   {all(pd.api.types.is_numeric_dtype(train_fe[c]) for c in train_fe.columns)}")



### Korelasi Fitur terhadap Target

In [ ]:
feat_cols = [c for c in train_fe.columns if c not in ("Target", "Date")]
corr = (train_fe[feat_cols + ["Target"]]
        .corr()["Target"].drop("Target")
        .sort_values(key=abs, ascending=False))
print(corr.round(5))

print("\nHEAD")
print(train_fe.head())


# 6. Outlier Handling

## 6.1 Fungsi & Penerapan Outlier Handling

In [ ]:
from math import isnan

def handle_outliers(df, column, method="iqr", action="cap", bounds=None):
    if column not in df.columns:
        raise KeyError(f"Column tidak ditemukan: {column}")
    if not pd.api.types.is_numeric_dtype(df[column]):
        raise TypeError(f"Column harus numerik: {column}")
    if method not in {"iqr", "zscore"}:
        raise ValueError("method harus 'iqr' atau 'zscore'")
    if action not in {"remove", "cap"}:
        raise ValueError("action harus 'remove' atau 'cap'")

    result = df.copy()
    values = result[column]

    # Hitung batas dari data saat ini, kecuali bounds diberikan dari train.
    if bounds is None:
        if method == "iqr":
            q1, q3 = values.quantile([0.25, 0.75])
            spread = q3 - q1
            lower = q1 - 1.5 * spread
            upper = q3 + 1.5 * spread
        else:
            mean = values.mean()
            std = values.std()
            lower = mean - 3 * std
            upper = mean + 3 * std
    else:
        lower, upper = bounds

    outlier_mask = values.notna() & ((values < lower) | (values > upper))
    before_rows = len(result)
    outlier_count = int(outlier_mask.sum())

    if action == "remove":
        result = result.loc[~outlier_mask].reset_index(drop=True)
        changed = outlier_count
        print(f"[{column}] {method} removal: {outlier_count} baris dihapus")
    else:
        result[column] = values.clip(lower=lower, upper=upper)
        changed = int((result[column] != values).fillna(False).sum())
        print(f"[{column}] {method} capping: {changed} nilai dibatasi")

    print(f"  Sebelum: {before_rows:,} baris | Sesudah: {len(result):,} baris")
    print(f"  Batas: [{lower:.6g}, {upper:.6g}] | Outlier terdeteksi: {outlier_count:,}")
    return result, (lower, upper)


In [ ]:
train_outlier = train_fe.copy()
test_outlier = test_fe.copy()

train_outlier, feature_bounds = handle_outliers(
    train_outlier,
    column="ret_1d",
    method="iqr",
    action="cap",
)

# Gunakan batas yang dipelajari dari train untuk test agar tidak terjadi leakage.
test_outlier, _ = handle_outliers(
    test_outlier,
    column="ret_1d",
    method="iqr",
    action="cap",
    bounds=feature_bounds,
)

train_removed, _ = handle_outliers(train_fe, "ret_1d", method="zscore", action="remove")

## 6.2 Visualisasi Outlier dan Hubungan Antar Fitur

In [ ]:
import matplotlib.pyplot as plt

plot_sample = train_fe.sample(n=min(50_000, len(train_fe)), random_state=42)

# 1. Heatmap korelasi fitur numerik terhadap fitur lainnya.
heatmap_cols = [
    "ret_1d", "ret_5d", "ret_20d", "intraday_range", "close_open",
    "high_close", "low_close", "log_volume", "volume_ratio",
    "vol_5d", "vol_20d", "ret_rank", "volume_rank", "Target",
]
heatmap_cols = [col for col in heatmap_cols if col in train_fe.columns]
corr_matrix = train_fe[heatmap_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
image = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(heatmap_cols)), heatmap_cols, rotation=75, ha="right")
ax.set_yticks(range(len(heatmap_cols)), heatmap_cols)
ax.set_title("Heatmap Korelasi Fitur Numerik")
fig.colorbar(image, ax=ax, label="Korelasi Pearson")
fig.tight_layout()
plt.show()


### Box Plot Deteksi Outlier per Fitur

In [ ]:
boxplot_cols = [
    "ret_1d", "ret_5d", "ret_20d", "intraday_range",
    "close_open", "volume_ratio", "vol_5d", "vol_20d", "Target",
]
boxplot_cols = [col for col in boxplot_cols if col in plot_sample.columns]

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, col in zip(axes.flat, boxplot_cols):
    ax.boxplot(plot_sample[col].dropna(), showfliers=True, vert=True)
    ax.set_title(col)
    ax.grid(axis="y", alpha=0.3)
for ax in axes.flat[len(boxplot_cols):]:
    ax.set_visible(False)
fig.suptitle("Box Plot Fitur untuk Deteksi Outlier", fontsize=15)
fig.tight_layout()
plt.show()

### Perbandingan ret_1d Sebelum vs Sesudah Capping

In [ ]:
if "train_outlier" in globals() and "ret_1d" in train_outlier.columns:
    compare_sample = train_fe[["ret_1d"]].sample(n=min(50_000, len(train_fe)), random_state=42)
    capped_sample = train_outlier.loc[compare_sample.index, "ret_1d"]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].boxplot(compare_sample["ret_1d"].dropna(), showfliers=True)
    axes[0].set_title("ret_1d Sebelum Capping")
    axes[1].boxplot(capped_sample.dropna(), showfliers=True)
    axes[1].set_title("ret_1d Sesudah Capping")
    for ax in axes:
        ax.grid(axis="y", alpha=0.3)
    fig.suptitle("Dampak Capping Outlier", fontsize=15)
    fig.tight_layout()
    plt.show()

# 7. Fitur Final

## 7.1 Rebuild dari frame clean

In [ ]:
train = train_clean.copy()
test  = test_clean.copy()

train = train.drop(columns=["_year"], errors="ignore")
test  = test.drop(columns=["_year"], errors="ignore")

train = train.sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)
test  = test.sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)

print(f"Base frame : train={train.shape}  test={test.shape}")
print(f"Kolom awal : {list(train.columns)}")

## 7.2 Fitur Temporal

In [ ]:
for df_ in (train, test):
    df_["year"]       = df_["Date"].dt.year
    df_["month"]      = df_["Date"].dt.month
    df_["day"]        = df_["Date"].dt.day
    df_["dayofweek"]  = df_["Date"].dt.dayofweek
    df_["quarter"]    = df_["Date"].dt.quarter
    df_["dayofyear"]  = df_["Date"].dt.dayofyear
    df_["weekofyear"] = df_["Date"].dt.isocalendar().week.astype(int)

print("Fitur temporal ditambahkan:",
      ["year", "month", "day", "dayofweek", "quarter", "dayofyear", "weekofyear"])

## 7.3 Encoding SecuritiesCode (Leak-free Target Encoding)

In [ ]:
K = 50
global_mean = train["Target"].mean()

# Out-of-fold encoding untuk train (setiap baris di-encode tanpa melihat targetnya sendiri)
train["SecuritiesCode_te"] = np.nan
kf = KFold(n_splits=5, shuffle=False)  # tanpa shuffle -> tetap menghormati urutan waktu
for tr_idx, va_idx in kf.split(train):
    sub = train.iloc[tr_idx]
    agg = sub.groupby("SecuritiesCode")["Target"].agg(["mean", "count"])
    smoothed = (agg["count"] * agg["mean"] + K * global_mean) / (agg["count"] + K)
    train.loc[va_idx, "SecuritiesCode_te"] = (
        train.iloc[va_idx]["SecuritiesCode"].map(smoothed).values
    )
train["SecuritiesCode_te"] = train["SecuritiesCode_te"].fillna(global_mean).astype(np.float32)

# Full-train mapping untuk test (kode saham yang tak terlihat di train -> global mean)
agg_full = train.groupby("SecuritiesCode")["Target"].agg(["mean", "count"])
smoothed_full = (agg_full["count"] * agg_full["mean"] + K * global_mean) / (agg_full["count"] + K)
test["SecuritiesCode_te"] = (
    test["SecuritiesCode"].map(smoothed_full).fillna(global_mean).astype(np.float32)
)

print(f"SecuritiesCode_te dibuat. Sebaran (train): "
      f"mean={train['SecuritiesCode_te'].mean():.6f}, std={train['SecuritiesCode_te'].std():.6f}")

## 7.4 Fitur Return, Volatilitas, dan Cross-sectional Rank

In [ ]:
train["_split"] = "train"
test["_split"]  = "test"
full = pd.concat([train, test], axis=0, ignore_index=True)
full = full.sort_values(["SecuritiesCode", "Date"]).reset_index(drop=True)
print(f"Concat: {full.shape}")

g = full.groupby("SecuritiesCode", sort=False)

# ---------- Returns (per-security, hanya pakai data historis) ----------
full["ret_1d"]  = g["Close"].pct_change()
full["ret_5d"]  = g["Close"].pct_change(5)
full["ret_20d"] = g["Close"].pct_change(20)

# ---------- Bentuk intraday (baris-per-baris, tidak bergantung waktu) ----------
full["intraday_range"] = (full["High"] - full["Low"]) / full["Close"]
full["close_open"]     = (full["Close"] - full["Open"]) / full["Open"]
full["high_close"]     = (full["High"] - full["Close"]) / full["Close"]
full["low_close"]      = (full["Close"] - full["Low"]) / full["Close"]

# ---------- Volume ----------
full["log_volume"]   = np.log1p(full["Volume"])
vol_ma20             = g["Volume"].transform(lambda s: s.rolling(20, min_periods=1).mean())
full["volume_ratio"] = full["Volume"] / vol_ma20
del vol_ma20

# ---------- Volatilitas ----------
full["vol_5d"]  = g["ret_1d"].transform(lambda s: s.rolling(5,  min_periods=2).std())
full["vol_20d"] = g["ret_1d"].transform(lambda s: s.rolling(20, min_periods=5).std())

# ---------- Rank cross-sectional (dalam Date yang sama) ----------
full["ret_rank"]    = full.groupby("Date")["ret_1d"].rank(pct=True)
full["volume_rank"] = full.groupby("Date")["Volume"].rank(pct=True)

print(f"Fitur return/volatilitas/rank ditambahkan. Shape full: {full.shape}")

## 7.5 Gabungkan & Finalisasi Feature Set

In [ ]:
# Drop OHLCV mentah (sudah digantikan fitur turunan di atas)
full = full.drop(columns=["Open", "High", "Low", "Close", "Volume"])

# Pisah kembali ke train / test
train_final = full[full["_split"] == "train"].drop(columns="_split").reset_index(drop=True)
test_final  = full[full["_split"] == "test"].drop(columns="_split").reset_index(drop=True)

# Buang baris dengan NaN di awal riwayat tiap saham (akibat pct_change/rolling window)
print(f"Sebelum drop NaN: train={train_final.shape}  test={test_final.shape}")
train_final = train_final.dropna().reset_index(drop=True)
test_final  = test_final.dropna().reset_index(drop=True)
print(f"Setelah drop NaN: train={train_final.shape}  test={test_final.shape}")

# Buang identifier yang tidak lagi diperlukan sebagai fitur model
train_final = train_final.drop(columns=["RowId", "SecuritiesCode", "Date"])
test_final  = test_final.drop(columns=["RowId", "SecuritiesCode", "Date"])

print(f"\nFitur final ({len(train_final.columns)} kolom):")
print(list(train_final.columns))
print(f"\nNaN tersisa -> train: {int(train_final.isna().sum().sum())} | "
      f"test: {int(test_final.isna().sum().sum())}")
print(f"\ntrain_final shape: {train_final.shape}")
print(f"test_final  shape: {test_final.shape}")
train_final.head()

# 8. Feature Scaling

## 8.1 Pisahkan Fitur dan Target

In [ ]:
FEATURE_COLS = [c for c in train_final.columns if c != "Target"]
TARGET_COL = "Target"

X_train = train_final[FEATURE_COLS].values.astype(np.float64)
y_train = train_final[TARGET_COL].values.astype(np.float64).reshape(-1, 1)

X_test = test_final[FEATURE_COLS].values.astype(np.float64)
y_test = test_final[TARGET_COL].values.astype(np.float64).reshape(-1, 1)

print(f"Jumlah fitur: {len(FEATURE_COLS)}")
print(f"Fitur       : {FEATURE_COLS}")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test : {X_test.shape} | y_test : {y_test.shape}")

## 8.2 Scaling untuk Pipeline SVR — StandardScaler 

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler_X_svr = StandardScaler()
scaler_y_svr = StandardScaler()

# PENTING: fit HANYA pada data train, lalu transform ke train & test.
# Statistik (mean, std) scaler tidak pernah "melihat" data test -> tidak ada leakage.
X_train_svr = scaler_X_svr.fit_transform(X_train)
X_test_svr  = scaler_X_svr.transform(X_test)

y_train_svr = scaler_y_svr.fit_transform(y_train).ravel()
y_test_svr  = scaler_y_svr.transform(y_test).ravel()

print("Pipeline SVR (StandardScaler, fit di Train saja):")
print(f"X_train_svr: {X_train_svr.shape} | X_test_svr: {X_test_svr.shape}")

## 8.3 Scaling untuk Pipeline LSTM — MinMaxScaler 

In [ ]:
scaler_X_lstm = MinMaxScaler()
scaler_y_lstm = MinMaxScaler()

X_train_lstm = scaler_X_lstm.fit_transform(X_train)
X_test_lstm  = scaler_X_lstm.transform(X_test)

y_train_lstm = scaler_y_lstm.fit_transform(y_train).ravel()
y_test_lstm  = scaler_y_lstm.transform(y_test).ravel()

print("Pipeline LSTM (MinMaxScaler, fit di Train saja):")
print(f"X_train_lstm: {X_train_lstm.shape} | X_test_lstm: {X_test_lstm.shape}")